In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import random

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

In [2]:
# 1. Facility Master (Czech Republic included, added Origin City)
# 1. Define the base facility information
facility_types = ['Plant', 'Port', 'Plant', 'Storage', 'Distribution Center', 'Distribution Center']


# 2. Helper function to assign capacity ranges based on your rules
def get_capacity(fac_type):
    if fac_type == 'Port':
        return np.random.randint(4500, 5500) # Around 5k
    elif fac_type == 'Plant':
        return np.random.randint(10000, 20000)
    elif fac_type in ['Storage', 'Distribution Center']:
        return np.random.randint(40000, 50000)
    return 0

# 3. Cost mapping dictionary based on your uploaded table
cost_mapping = {
    'Port': {'Storage': 300000, 'Handling': 12.00, 'Overhead': 100000},
    'Plant': {'Storage': 800000, 'Handling': 6.50, 'Overhead': 250000},
    'Storage': {'Storage': 1200000, 'Handling': 4.50, 'Overhead': 150000},
    'Distribution Center': {'Storage': 1500000, 'Handling': 5.50, 'Overhead': 350000}
}

# 4. Generate the dataframe
facilities = pd.DataFrame({
    'FacilityID': ['F01', 'F02', 'F03', 'F04', 'F05', 'F06'],
    'FacilityName': ['Bavaria Dairy Plant', 'Maasvlakte Port DC', 'Laval Dairy Plant', 'Cremona Storage Hub', 'Brno South Logistics', 'Frankfurt Central DC'],
    'FacilityType': facility_types,
    'OriginCity': ['Wasserburg', 'Rotterdam', 'Laval', 'Cremona', 'Pohořelice', 'Frankfurt'],
    'Country': ['Germany', 'Netherlands', 'France', 'Italy', 'Czech Republic', 'Germany'],
    
    # ---------------------------------------------------------
    # NEW: Latitude & Longitude included for all 6 facilities
    # ---------------------------------------------------------
    'Latitude': [48.0610, 51.9540, 48.0650, 45.1230, 48.9750, 50.1109],   
    'Longitude': [12.2280, 4.0530, -0.7510, 10.0080, 16.5180, 8.6821],
    
    # Apply the capacity function and cost mappings via list comprehensions
    'CapacityInPallet': [get_capacity(ft) for ft in facility_types],
    'AnnualStorageCost': [cost_mapping[ft]['Storage'] for ft in facility_types],
    'HandlingCostPerPallet': [cost_mapping[ft]['Handling'] for ft in facility_types],
    'TotalOverheadCost': [cost_mapping[ft]['Overhead'] for ft in facility_types]
})

In [3]:
# 2. Customer Master (Now mapped to specific Cities)
eu_countries = ['Germany', 'France', 'Italy', 'Spain', 'Netherlands', 'Belgium', 'Czech Republic', 'Sweden']
country_weights = [0.28, 0.22, 0.18, 0.12, 0.08, 0.05, 0.04, 0.03] 

city_coords = {
    'Germany': {'Berlin': (52.52, 13.40), 'Munich': (48.13, 11.58), 'Hamburg': (53.55, 9.99), 'Frankfurt': (50.11, 8.68)},
    'France': {'Paris': (48.85, 2.35), 'Lyon': (45.76, 4.83), 'Marseille': (43.29, 5.36), 'Toulouse': (43.60, 1.44)},
    'Italy': {'Rome': (41.90, 12.49), 'Milan': (45.46, 9.19), 'Naples': (40.85, 14.26), 'Turin': (45.07, 7.68)},
    'Spain': {'Madrid': (40.41, -3.70), 'Barcelona': (41.38, 2.16), 'Valencia': (39.46, -0.37), 'Seville': (37.38, -5.98)},
    'Netherlands': {'Amsterdam': (52.36, 4.90), 'Rotterdam': (51.92, 4.47), 'The Hague': (52.07, 4.30), 'Utrecht': (52.09, 5.12)},
    'Belgium': {'Brussels': (50.85, 4.35), 'Antwerp': (51.21, 4.40), 'Ghent': (51.05, 3.71), 'Liege': (50.63, 5.57)},
    'Czech Republic': {'Prague': (50.07, 14.43), 'Brno': (49.19, 16.60), 'Ostrava': (49.82, 18.26), 'Plzen': (49.73, 13.37)},
    'Sweden': {'Stockholm': (59.32, 18.06), 'Gothenburg': (57.70, 11.97), 'Malmo': (55.60, 13.00), 'Uppsala': (59.85, 17.63)}
}

dairy_prefixes = ['Alpine', 'Nordic', 'Euro', 'Pure', 'Green', 'Valley', 'Golden']
dairy_suffixes = ['Dairy', 'Foods', 'Milk Co.', 'Nutrition', 'Brands']
channels = ['Retail', 'Foodservice', 'Wholesale', 'Manufacturing', 'Key Account']

customers = []
for i in range(1, 251): 
    account_name = f"{random.choice(dairy_prefixes)} {random.choice(dairy_suffixes)}"
    # Apply population weights here
    country = np.random.choice(eu_countries, p=country_weights)
    city = random.choice(list(city_coords[country].keys()))
    
    base_lat, base_lon = city_coords[country][city]
    
    customers.append({
        'ShipToID': f"ST{i:04d}",
        'ShipToParty': f"{account_name} - {city}",
        'Account': account_name,
        'Channel': np.random.choice(channels, p=[0.4, 0.25, 0.15, 0.1, 0.1]),
        # 'ABCClass': np.random.choice(['A', 'B', 'C'], p=[0.2, 0.3, 0.5]),
        'Address': f"{np.random.randint(1, 999)} Commerce St",
        'City': city,
        'ShipToCountry': country,
        'Continent': 'Europe',
        'Latitude': round(base_lat + np.random.uniform(-0.05, 0.05), 5),
        'Longitude': round(base_lon + np.random.uniform(-0.05, 0.05), 5)
    })
customer_master = pd.DataFrame(customers)

In [4]:
# 3. Product Master
# Category setup
category_distribution = (['Cheese'] * 35) + (['Butter'] * 30) + (['UHT'] * 25) + (['Powder'] * 10)
random.shuffle(category_distribution)

# 1. Define Price ranges per KG (Powder > Cheese > Butter > UHT)
price_per_kg_ranges = {
    'Powder': (10.0, 15.0),  # Premium priced
    'Cheese': (7.0, 10.0),   # Value-add
    'Butter': (4.5, 7.0),    # High raw material dependence
    'UHT':    (0.8, 1.5)     # Liquid, heavy, volume business
}

# 2. Define COGS % of Sales Price (Based on milk volume & production time)
cogs_margin_ranges = {
    'Powder': (0.80, 0.85),  # High energy cost for spray drying
    'Cheese': (0.78, 0.88),  # Best margin due to aging/value-add
    'Butter': (0.85, 0.9),  # High COGS due to taking 20L milk per KG
    'UHT':    (0.88, 0.95)   # Tightest margin, fast commodity
}

products = []
for i in range(1, 101):
    cat = category_distribution[i-1]
    kg_pallet = np.random.choice([500, 800, 1000])
    sales_unit = 'Carton' if cat in ['UHT', 'Butter', 'Cheese'] else 'Bag'
    kg_su = 10 if sales_unit == 'Carton' else 25
    
    # Calculate realistic Price
    base_price_per_kg = np.random.uniform(*price_per_kg_ranges[cat])
    price = base_price_per_kg * kg_su
    
    # Calculate realistic COGS based on category economics
    cogs_percentage = np.random.uniform(*cogs_margin_ranges[cat])
    cogs = price * cogs_percentage
    
    # Optional realism: Shelf life tuning based on category
    if cat == 'UHT': shelf_life = np.random.choice([180, 365])
    elif cat == 'Powder': shelf_life = np.random.choice([365, 730])
    elif cat == 'Cheese': shelf_life = np.random.choice([90, 180, 365])
    else: shelf_life = np.random.choice([90, 180]) # Butter
    
    products.append({
        'ProductID': f"P{i:03d}",
        'ProductName': f"Premium {cat} Variant {i}",
        'Category': cat,
        'KGPerPallet': kg_pallet,
        'SalesUnit': sales_unit,
        'KGPerSalesUnit': kg_su,
        'Price': round(price, 2),
        'COGS': round(cogs, 2),
        'ShelfLife': int(shelf_life)
    })

product_master = pd.DataFrame(products)

In [5]:
# Helper function to calculate distance between two coordinates in kilometers
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0 # Earth radius in km
    dLat = np.radians(lat2 - lat1)
    dLon = np.radians(lon2 - lon1)
    a = (np.sin(dLat / 2) ** 2 +
         np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dLon / 2) ** 2)
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

In [6]:
# 4. Transport Rate Card (By City/Facility, Distance-based formula)
# Base rate per pallet per KM
BASE_RATE_PER_KM = 0.1 
# Fixed additional rates reflecting labor/fuel indices in origin countries
additional_rates = {
    'Germany': 25.0,
    'Netherlands': 28.0,
    'France': 22.0,
    'Italy': 20.0,
    'Czech Republic': 12.0
}

rates = []

# --- PART 1: Rates to Customers ---
unique_destinations = customer_master.groupby('City').agg({'Latitude':'mean', 'Longitude':'mean', 'ShipToCountry':'first'}).reset_index()

for _, fac in facilities.iterrows():
    for _, dest in unique_destinations.iterrows():
        distance_km = haversine_distance(fac['Latitude'], fac['Longitude'], dest['Latitude'], dest['Longitude'])
        origin_country = fac['Country']
        add_rate = additional_rates.get(origin_country, 15.0)
        cost_per_pallet = (BASE_RATE_PER_KM * distance_km) + add_rate
        
        rates.append({
            'OriginID': fac['FacilityID'],
            'OriginCity': fac['OriginCity'],
            'OriginCountry': origin_country,
            'DestinationID': 'CUSTOMER', # Generic flag for customer cities
            'DestinationCity': dest['City'],
            'DestinationCountry': dest['ShipToCountry'],
            'DistanceKM': round(distance_km, 2),
            'CostPerPallet': round(cost_per_pallet, 2),
            'RouteType': 'Outbound to Customer'
        })

# --- PART 2: Rates for Internal Transfers (Facility to Facility) ---
for _, origin_fac in facilities.iterrows():
    for _, dest_fac in facilities.iterrows():
        if origin_fac['FacilityID'] != dest_fac['FacilityID']: # Don't calculate rate to itself
            distance_km = haversine_distance(origin_fac['Latitude'], origin_fac['Longitude'], dest_fac['Latitude'], dest_fac['Longitude'])
            origin_country = origin_fac['Country']
            add_rate = additional_rates.get(origin_country, 15.0)
            
            # Internal transfers usually benefit from full-truckload (FTL) bulk discounts. 
            # We can multiply the base cost by 0.8 to simulate a 20% bulk discount.
            cost_per_pallet = ((BASE_RATE_PER_KM * distance_km) + add_rate) * 0.8
            
            rates.append({
                'OriginID': origin_fac['FacilityID'],
                'OriginCity': origin_fac['OriginCity'],
                'OriginCountry': origin_country,
                'DestinationID': dest_fac['FacilityID'], # Explicitly link to the destination Facility ID
                'DestinationCity': dest_fac['OriginCity'],
                'DestinationCountry': dest_fac['Country'],
                'DistanceKM': round(distance_km, 2),
                'CostPerPallet': round(cost_per_pallet, 2),
                'RouteType': 'Internal STO'
            })

transport_rates = pd.DataFrame(rates)

In [7]:
num_orders = 200000 

# Map destination countries to their closest, logical fulfilling facilities
fulfillment_network = {
    'Germany': ['F01', 'F06'],       # Bavaria Plant & Frankfurt DC
    'France': ['F03'],               # Laval Plant
    'Italy': ['F04'],                # Cremona Storage
    'Spain': ['F03', 'F04'],         # Split between Laval and Cremona
    'Netherlands': ['F02'],          # Rotterdam Port
    'Belgium': ['F02', 'F03'],       # Split between Rotterdam and Laval
    'Czech Republic': ['F05'],       # Brno DC
    'Sweden': ['F02', 'F06']         # Served via Rotterdam Port or Frankfurt DC
}

# When building Sales Orders, pick the facility based on the Customer's country
def assign_local_facility(country):
    return random.choice(fulfillment_network[country])


# A. Build Customer Assortments (Catalogs)
taste_matrix = {
    'France': [0.40, 0.45, 0.10, 0.05], 'Italy': [0.55, 0.10, 0.30, 0.05],
    'Germany': [0.35, 0.30, 0.25, 0.10], 'Netherlands': [0.60, 0.15, 0.15, 0.10],
    'Spain': [0.30, 0.10, 0.55, 0.05], 'Belgium': [0.35, 0.35, 0.20, 0.10],
    'Czech Republic': [0.35, 0.30, 0.25, 0.10], 'Sweden': [0.30, 0.25, 0.40, 0.05]
}
categories_list = ['Cheese', 'Butter', 'UHT', 'Powder']
cat_to_prods = product_master.groupby('Category')['ProductID'].apply(list).to_dict()

customer_catalogs = {}
for _, cust in customer_master.iterrows():
    catalog_size = np.random.randint(5, 16) 
    cust_catalog = set()
    while len(cust_catalog) < catalog_size:
        chosen_cat = np.random.choice(categories_list, p=taste_matrix[cust['ShipToCountry']])
        cust_catalog.add(random.choice(cat_to_prods[chosen_cat]))
    customer_catalogs[cust['ShipToID']] = list(cust_catalog)

# B. Generate Balanced Dates
months = pd.date_range(start="2024-01-01", end="2025-12-31", freq='MS')
orders_per_month = num_orders // len(months)
balanced_dates = []
for month_start in months:
    days = pd.date_range(start=month_start, end=month_start + pd.offsets.MonthEnd(0))
    balanced_dates.extend(np.random.choice(days, orders_per_month))
if (num_orders % len(months)) > 0:
    balanced_dates.extend(np.random.choice(pd.date_range(start="2024-01-01", end="2025-12-31"), num_orders % len(months)))
np.random.shuffle(balanced_dates)

# C. Build Base Orders
sales_orders = pd.DataFrame({
    'OrderID': [f"SO{i:06d}" for i in range(1, num_orders + 1)],
    'FacilityID': np.random.choice(facilities['FacilityID'], num_orders),
    'ShipToID': np.random.choice(customer_master['ShipToID'], num_orders),
    'Date': balanced_dates,  
    'SalesVolume': np.random.randint(10, 500, num_orders)
})

# D. Assign Products strictly from Catalogs
def assign_from_catalog(ship_to): return random.choice(customer_catalogs[ship_to])
sales_orders['ProductID'] = sales_orders['ShipToID'].apply(assign_from_catalog)

# E. Dynamic Pricing Logic
sales_orders = sales_orders.merge(customer_master[['ShipToID', 'Channel', 'ShipToCountry']], on='ShipToID', how='left')
sales_orders = sales_orders.merge(product_master[['ProductID', 'Price']], on='ProductID', how='left')

channel_pricing = {'Key Account': 0.90, 'Wholesale': 0.95, 'Manufacturing': 0.98, 'Foodservice': 1.02, 'Retail': 1.06}
sales_orders['CustMult'] = sales_orders['Channel'].map(channel_pricing).fillna(1.0)

sales_orders['Date'] = pd.to_datetime(sales_orders['Date'])
seasonality = {1: 1.05, 2: 1.03, 3: 0.98, 4: 0.95, 5: 0.95, 6: 0.98, 7: 1.00, 8: 1.00, 9: 1.02, 10: 1.04, 11: 1.05, 12: 1.06}
sales_orders['SeasMult'] = sales_orders['Date'].dt.month.map(seasonality)

sales_orders['Noise'] = np.random.uniform(0.97, 1.03, len(sales_orders))
sales_orders['ActualPrice'] = (sales_orders['Price'] * sales_orders['CustMult'] * sales_orders['SeasMult'] * sales_orders['Noise']).round(2)

# Replace the random FacilityID assignment with this:
sales_orders['FacilityID'] = sales_orders['ShipToCountry'].apply(assign_local_facility)

sales_orders.drop(columns=['Channel', 'Price', 'CustMult', 'SeasMult', 'Noise'], inplace=True)

In [8]:
print("Generating Internal Transfers (Plant -> DC)...")

# 1. Define the internal supply network (Who supplies who?)
# F01 (Bavaria Plant) supplies Eastern/Central Europe DCs
# F03 (Laval Plant) supplies Western/Southern Europe DCs
internal_network = {
    'F02': 'F03',  # Rotterdam Port supplied by Laval
    'F04': 'F03',  # Cremona Storage supplied by Laval
    'F05': 'F01',  # Brno DC supplied by Bavaria
    'F06': 'F01'   # Frankfurt DC supplied by Bavaria
}

sales_merged = sales_orders.merge(
    product_master[['ProductID', 'KGPerSalesUnit', 'KGPerPallet']], 
    on='ProductID', 
    how='left'
)

# 2. Calculate the total physical weight of the order
sales_merged['SalesVolumeKG'] = sales_merged['SalesVolume'] * sales_merged['KGPerSalesUnit']

# 3. Calculate exact pallets shipped per order
sales_merged['SalesPallets'] = sales_merged['SalesVolumeKG'] / sales_merged['KGPerPallet']

# Ensure the Date column is datetime
sales_merged['Date'] = pd.to_datetime(sales_merged['Date'])

sales_merged['MonthEnd'] = sales_merged['Date'] + pd.offsets.MonthEnd(0)
monthly_demand = sales_merged[sales_merged['FacilityID'].isin(internal_network.keys())].groupby(
    ['MonthEnd', 'FacilityID', 'ProductID']
)['SalesPallets'].sum().reset_index()

# 3. Generate Bulk Stock Transfer Orders (STOs)
sto_records = []
sto_counter = 1

for _, row in monthly_demand.iterrows():
    receiving_facility = row['FacilityID']
    supplying_plant = internal_network[receiving_facility]
    
    # Send enough pallets to cover sales + a small random safety stock buffer (5% to 15%)
    transfer_pallets = np.ceil(row['SalesPallets'] * np.random.uniform(1.05, 1.15))
    
    # Ship the goods 1 to 3 weeks BEFORE the month they are needed to sell
    days_to_subtract = np.random.randint(7, 21)
    ship_date = row['MonthEnd'] - pd.Timedelta(days=days_to_subtract)
    
    sto_records.append({
        'TransferID': f"STO{sto_counter:05d}",
        'Date': ship_date,
        'OriginFacilityID': supplying_plant,
        'DestinationFacilityID': receiving_facility,
        'ProductID': row['ProductID'],
        'TransferPallets': transfer_pallets,
        'SupplyChainFlow': 'EU to EU Internal' # Perfect for your Sankey Chart!
    })
    sto_counter += 1

internal_transfers = pd.DataFrame(sto_records)

Generating Internal Transfers (Plant -> DC)...


In [9]:
# 6. Inventory (Weekly snapshot)
# Merge with Product Master to get unit weights
sales_merged = sales_orders.merge(product_master[['ProductID', 'KGPerSalesUnit', 'KGPerPallet']], on='ProductID', how='left')

# Calculate exact pallets shipped per order
sales_merged['SalesVolumeKG'] = sales_merged['SalesVolume'] * sales_merged['KGPerSalesUnit']
sales_merged['SalesPallets'] = sales_merged['SalesVolumeKG'] / sales_merged['KGPerPallet']

# Ensure the Date column is datetime
sales_merged['Date'] = pd.to_datetime(sales_merged['Date'])

# 2. Aggregate Sales by Week, Facility, and Product
# freq='W' automatically groups daily sales into weekly buckets (ending on Sunday)
weekly_sales = sales_merged.groupby([
    pd.Grouper(key='Date', freq='W'), 
    'FacilityID', 
    'ProductID'
])['SalesPallets'].sum().reset_index()

weekly_sales.rename(columns={'Date': 'Week'}, inplace=True)

# 3. Generate the Inventory matching the Sales + Facility Capacity
weekly_dates = pd.date_range(start="2024-01-01", end="2025-12-31", freq='W')
capacity_dict = facilities.set_index('FacilityID')['CapacityInPallet'].to_dict()

inventory_records = []

for current_week in weekly_dates:
    # Filter sales that happened in this specific week
    week_sales = weekly_sales[weekly_sales['Week'] == current_week]
    
    for fac_id, capacity in capacity_dict.items():
        # Filter sales for this facility this week
        fac_week_sales = week_sales[week_sales['FacilityID'] == fac_id]
        
        # Determine target total pallets for this facility (60% to 105% capacity)
        target_utilization = np.random.uniform(0.60, 1.05)
        target_total_pallets = capacity * target_utilization
        
        base_inventory = {}
        total_required_pallets = 0
        
        # Step A: Secure inventory for all products that were sold this week
        if not fac_week_sales.empty:
            for _, row in fac_week_sales.iterrows():
                # Round up to ensure we have enough whole pallets to cover the sales
                required_pallets = np.ceil(row['SalesPallets'])
                base_inventory[row['ProductID']] = required_pallets
                total_required_pallets += required_pallets
        
        # Step B: Fill the remaining facility capacity with random inventory
        remaining_pallets = target_total_pallets - total_required_pallets
        
        if remaining_pallets > 0:
            sold_products = list(base_inventory.keys())
            other_products = list(set(product_master['ProductID']) - set(sold_products))
            
            # Pick additional random products so the warehouse has ~30-60 active SKUs
            num_additional = max(0, np.random.randint(30, 61) - len(sold_products))
            num_additional = min(num_additional, len(other_products))
            
            additional_products = np.random.choice(other_products, size=num_additional, replace=False).tolist() if num_additional > 0 else []
            all_active_products = sold_products + additional_products
            
            # Distribute the remaining pallets mathematically across these active products
            weights = np.random.dirichlet(np.ones(len(all_active_products)), size=1)[0]
            distributed_volumes = np.round(weights * remaining_pallets).astype(int)
            
            for prod, extra_vol in zip(all_active_products, distributed_volumes):
                base_inventory[prod] = base_inventory.get(prod, 0) + extra_vol

        # Append to the final inventory records
        for prod, vol in base_inventory.items():
            # Ensure no zeroes accidentally make it into the inventory table
            final_vol = max(int(vol), 1)
            
            inventory_records.append({
                'Date': current_week,
                'FacilityID': fac_id,
                'ProductID': prod,
                'VolumeInPallet': final_vol
            })

inventory = pd.DataFrame(inventory_records)

In [10]:
# import matplotlib.pyplot as plt

# # 1. Merge the data to get Country, Category, and KG conversion rate
# # Assuming you have already loaded: sales_orders, customer_master, product_master
# df = sales_orders.merge(customer_master[['ShipToID', 'ShipToCountry']], on='ShipToID', how='left')
# df = df.merge(product_master[['ProductID', 'Category', 'KGPerSalesUnit']], on='ProductID', how='left')

# # 2. Calculate volume in KG
# df['SalesVolumeKG'] = df['SalesVolume'] * df['KGPerSalesUnit']

# # 3. Group by Country and Category, then unstack to create category columns
# grouped = df.groupby(['ShipToCountry', 'Category'])['SalesVolumeKG'].sum().unstack(fill_value=0)

# # 4. Calculate a 'Total' to sort the bars descending, then drop it so it doesn't plot as a category
# grouped['Total'] = grouped.sum(axis=1)
# grouped = grouped.sort_values('Total', ascending=False).drop(columns='Total')

# # 5. Create stacked bar chart
# fig, ax = plt.subplots(figsize=(12, 7))

# # Custom distinct colors for the dairy categories
# colors = ['#4c72b0', '#dd8452', '#55a868', '#c44e52']
# grouped.plot(kind='bar', stacked=True, ax=ax, color=colors[:len(grouped.columns)], edgecolor='black', width=0.75)

# # Formatting
# ax.set_title('Total Sales Volume (KG) by Country and Category (2025)', fontsize=15, fontweight='bold')
# ax.set_xlabel('Country', fontsize=12, fontweight='bold')
# ax.set_ylabel('Total Sales Volume (KG)', fontsize=12, fontweight='bold')
# plt.xticks(rotation=45, ha='right')

# # Format y-axis to show commas (e.g., 1,000,000 instead of 1000000)
# ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
# ax.legend(title='Product Category', title_fontsize='11', fontsize='10', loc='upper right', framealpha=0.9)

# # Add totals on top of each stacked bar
# totals = grouped.sum(axis=1)
# for i, total in enumerate(totals):
#     ax.text(i, total + (total * 0.015), f"{int(total):,}", ha='center', va='bottom', fontsize=10, fontweight='bold')

# plt.tight_layout()
# plt.savefig('sales_volume_kg_stacked.png', dpi=150)
# plt.show()

In [11]:
df = sales_orders.merge(product_master, on='ProductID', how='left')
df = df.merge(customer_master, on='ShipToID', how='left')
df = df.rename(columns={
'Address': 'Address_ShipTo',
'City': 'City_ShipTo',
'Continent': 'Continent_ShipTo',
'Latitude': 'Latitude_ShipTo',
'Longitude': 'Longitude_ShipTo'
})
df.head()

,OrderID,FacilityID,ShipToID,Date,SalesVolume,ProductID,ShipToCountry_x,ActualPrice,ProductName,Category,...,ShelfLife,ShipToParty,Account,Channel,Address_ShipTo,City_ShipTo,ShipToCountry_y,Continent_ShipTo,Latitude_ShipTo,Longitude_ShipTo
0,SO000001,F04,ST0195,2025-08-10,147,P077,Italy,80.84,Premium Cheese Variant 77,Cheese,...,180,Nordic Foods - Milan,Nordic Foods,Retail,229 Commerce St,Milan,Italy,Europe,45.42785,9.20527
1,SO000002,F01,ST0181,2024-11-14,278,P100,Germany,68.04,Premium Butter Variant 100,Butter,...,90,Golden Nutrition - Munich,Golden Nutrition,Wholesale,666 Commerce St,Munich,Germany,Europe,48.11335,11.56982
2,SO000003,F03,ST0106,2024-09-01,59,P090,France,54.02,Premium Butter Variant 90,Butter,...,180,Nordic Milk Co. - Toulouse,Nordic Milk Co.,Key Account,566 Commerce St,Toulouse,France,Europe,43.60843,1.48012
3,SO000004,F02,ST0206,2024-11-29,223,P037,Sweden,10.39,Premium UHT Variant 37,UHT,...,365,Pure Foods - Malmo,Pure Foods,Foodservice,109 Commerce St,Malmo,Sweden,Europe,55.61912,13.01832
4,SO000005,F06,ST0216,2025-06-02,91,P004,Germany,272.28,Premium Powder Variant 4,Powder,...,365,Euro Milk Co. - Frankfurt,Euro Milk Co.,Foodservice,365 Commerce St,Frankfurt,Germany,Europe,50.07547,8.69233


In [12]:
df = df.merge(facilities, on='FacilityID', how='left')
df = df.rename(columns={
    'Address': 'Address_Facility',
    'OriginCity': 'City_Facility',
    'Country': 'Country_Facility',
    # 'Continent': 'Continent_ShipTo',
    'Latitude': 'Latitude_Facility',
    'Longitude': 'Longitude_Facility'
})
df.head()

,OrderID,FacilityID,ShipToID,Date,SalesVolume,ProductID,ShipToCountry_x,ActualPrice,ProductName,Category,...,FacilityName,FacilityType,City_Facility,Country_Facility,Latitude_Facility,Longitude_Facility,CapacityInPallet,AnnualStorageCost,HandlingCostPerPallet,TotalOverheadCost
0,SO000001,F04,ST0195,2025-08-10,147,P077,Italy,80.84,Premium Cheese Variant 77,Cheese,...,Cremona Storage Hub,Storage,Cremona,Italy,45.1230,10.0080,45390,1200000,4.5,150000
1,SO000002,F01,ST0181,2024-11-14,278,P100,Germany,68.04,Premium Butter Variant 100,Butter,...,Bavaria Dairy Plant,Plant,Wasserburg,Germany,48.0610,12.2280,17270,800000,6.5,250000
2,SO000003,F03,ST0106,2024-09-01,59,P090,France,54.02,Premium Butter Variant 90,Butter,...,Laval Dairy Plant,Plant,Laval,France,48.0650,-0.7510,10860,800000,6.5,250000
3,SO000004,F02,ST0206,2024-11-29,223,P037,Sweden,10.39,Premium UHT Variant 37,UHT,...,Maasvlakte Port DC,Port,Rotterdam,Netherlands,51.9540,4.0530,4935,300000,12.0,100000
4,SO000005,F06,ST0216,2025-06-02,91,P004,Germany,272.28,Premium Powder Variant 4,Powder,...,Frankfurt Central DC,Distribution Center,Frankfurt,Germany,50.1109,8.6821,45734,1500000,5.5,350000


In [13]:
transport_rates.columns

Index(['OriginID', 'OriginCity', 'OriginCountry', 'DestinationID',
       'DestinationCity', 'DestinationCountry', 'DistanceKM', 'CostPerPallet',
       'RouteType'],
      dtype='object')

In [14]:
df = df.merge(transport_rates, left_on=['City_Facility', 'City_ShipTo'], right_on=['OriginCity', 'DestinationCity'], how='left')
df.head()

,OrderID,FacilityID,ShipToID,Date,SalesVolume,ProductID,ShipToCountry_x,ActualPrice,ProductName,Category,...,TotalOverheadCost,OriginID,OriginCity,OriginCountry,DestinationID,DestinationCity,DestinationCountry,DistanceKM,CostPerPallet,RouteType
0,SO000001,F04,ST0195,2025-08-10,147,P077,Italy,80.84,Premium Cheese Variant 77,Cheese,...,150000,F04,Cremona,Italy,CUSTOMER,Milan,Italy,73.54,27.35,Outbound to Customer
1,SO000002,F01,ST0181,2024-11-14,278,P100,Germany,68.04,Premium Butter Variant 100,Butter,...,250000,F01,Wasserburg,Germany,CUSTOMER,Munich,Germany,48.26,29.83,Outbound to Customer
2,SO000003,F03,ST0106,2024-09-01,59,P090,France,54.02,Premium Butter Variant 90,Butter,...,250000,F03,Laval,France,CUSTOMER,Toulouse,France,523.87,74.39,Outbound to Customer
3,SO000004,F02,ST0206,2024-11-29,223,P037,Sweden,10.39,Premium UHT Variant 37,UHT,...,100000,F02,Rotterdam,Netherlands,CUSTOMER,Malmo,Sweden,715.65,99.57,Outbound to Customer
4,SO000005,F06,ST0216,2025-06-02,91,P004,Germany,272.28,Premium Powder Variant 4,Powder,...,350000,F06,Frankfurt,Germany,CUSTOMER,Frankfurt,Germany,0.79,25.08,Outbound to Customer


In [15]:
# ... (Assuming df already contains the base sales orders merged with product, customer, rates, and facilities) ...

df['SalesVolumeKG'] = df['SalesVolume'] * df['KGPerSalesUnit']
df['QuantityInMT'] = df['SalesVolumeKG'] / 1000
df['RawPallets'] = df['SalesVolumeKG'] / df['KGPerPallet']
df['PalletsShipped'] = np.ceil(df['RawPallets'] * 2) / 2  # Round to nearest 0.5

# --- Logistics Costs (Primary Outbound) ---
# Renamed to OutboundTransportCost to clearly separate from total transport cost
df['OutboundTransportCost'] = df['PalletsShipped'] * df['CostPerPallet']
df['TotalHandlingCost'] = df['PalletsShipped'] * df['HandlingCostPerPallet']

# ==============================================================================
# NEW STEP: Calculate Internal Transfer Transport Cost (Plant -> DC)
# ==============================================================================
# 1. Merge internal transfers with the rate card to get the cost per pallet
internal_transfers = internal_transfers.merge(
    transport_rates[['OriginID', 'DestinationID', 'CostPerPallet']],
    left_on=['OriginFacilityID', 'DestinationFacilityID'],
    right_on=['OriginID', 'DestinationID'],
    how='left'
)
internal_transfers.rename(columns={'CostPerPallet': 'TransferCostPerPallet'}, inplace=True)

# 2. Calculate total cost for the transfer
internal_transfers['TransferTransportCost'] = internal_transfers['TransferPallets'] * internal_transfers['TransferCostPerPallet']

# 3. Aggregate these costs by Receiving Facility (Destination) and Product
transfer_costs_agg = internal_transfers.groupby(['DestinationFacilityID', 'ProductID'])['TransferTransportCost'].sum().reset_index()
# Rename DestinationFacilityID to FacilityID so it matches the Sales Order dataframe for merging
transfer_costs_agg.rename(columns={'DestinationFacilityID': 'FacilityID', 'TransferTransportCost': 'TotalTransferCost_FacProd'}, inplace=True)
# ==============================================================================


# --- Fixed Cost Allocation & IOC ---
# Step A: Avg Inventory
avg_inv = inventory.groupby(['FacilityID', 'ProductID'])['VolumeInPallet'].mean().reset_index(name='AvgInventoryInPallet')
fac_total = avg_inv.groupby('FacilityID')['AvgInventoryInPallet'].sum().reset_index(name='FacTotalAvgInventory')

alloc_df = avg_inv.merge(fac_total, on='FacilityID')
alloc_df['InvAllocRatio'] = alloc_df['AvgInventoryInPallet'] / alloc_df['FacTotalAvgInventory']
alloc_df['InvAllocRatio'] = alloc_df['InvAllocRatio'].fillna(0)

# Step B: Allocate to Facility-Product Level
alloc_df = alloc_df.merge(facilities[['FacilityID', 'AnnualStorageCost', 'TotalOverheadCost']], on='FacilityID')
alloc_df['StorageCost_FacProd'] = alloc_df['AnnualStorageCost'] * 2 * alloc_df['InvAllocRatio']
alloc_df['OverheadCost_FacProd'] = alloc_df['TotalOverheadCost'] * 2 * alloc_df['InvAllocRatio']

alloc_df = alloc_df.merge(product_master[['ProductID', 'COGS', 'KGPerPallet', 'KGPerSalesUnit']], on='ProductID')
alloc_df['COGSPerPallet'] = alloc_df['COGS'] * (alloc_df['KGPerPallet'] / alloc_df['KGPerSalesUnit'])
alloc_df['IOC_FacProd'] = 0.10 * alloc_df['AvgInventoryInPallet'] * alloc_df['COGSPerPallet']

# Step C: Allocate to Sales Order Row Level
sales_totals = df.groupby(['FacilityID', 'ProductID'])['SalesVolumeKG'].sum().reset_index(name='TotalSalesVolumeKG_FacProd')

# Merge sales totals AND the new transfer costs into alloc_df
alloc_df = alloc_df.merge(sales_totals, on=['FacilityID', 'ProductID'], how='left')
alloc_df = alloc_df.merge(transfer_costs_agg, on=['FacilityID', 'ProductID'], how='left')

# Merge all allocated facility-level costs into the main sales dataframe
df = df.merge(alloc_df[['FacilityID', 'ProductID', 'StorageCost_FacProd', 'OverheadCost_FacProd', 'IOC_FacProd', 'TotalTransferCost_FacProd', 'TotalSalesVolumeKG_FacProd']], 
              on=['FacilityID', 'ProductID'], how='left')

# Fill NaNs for facilities/products that didn't have any transfers or inventory
df.fillna({
    'StorageCost_FacProd': 0, 
    'OverheadCost_FacProd': 0, 
    'IOC_FacProd': 0, 
    'TotalTransferCost_FacProd': 0, 
    'TotalSalesVolumeKG_FacProd': 1
}, inplace=True)

# Calculate the Row Allocation Ratio based on Sales Volume
df['RowAllocRatio'] = df['SalesVolumeKG'] / df['TotalSalesVolumeKG_FacProd']
df['RowAllocRatio'] = df['RowAllocRatio'].replace([np.inf, -np.inf, np.nan], 0)

# Apply the ratio to the allocated buckets
df['StorageCostAllocated'] = df['StorageCost_FacProd'] * df['RowAllocRatio']
df['OverheadCostAllocated'] = df['OverheadCost_FacProd'] * df['RowAllocRatio']
df['TotalStorageCost'] = df['StorageCostAllocated'] + df['OverheadCostAllocated']
df['IOC'] = df['IOC_FacProd'] * df['RowAllocRatio']

# Apply the ratio to the internal transfer transport costs
df['TransferTransportAllocated'] = df['TotalTransferCost_FacProd'] * df['RowAllocRatio']

# Calculate Final Total Transport Cost
df['TotalTransportCost'] = df['OutboundTransportCost'] + df['TransferTransportAllocated']


# --- Financial Waterfall ---
# Assuming ActualPrice from earlier dynamic pricing generation
df['TotalSalesValue'] = df['SalesVolume'] * df['Price'] 
df['TotalCOGS'] = df['SalesVolume'] * df['COGS']

df['TotalS&D'] = df['TotalTransportCost'] + df['TotalHandlingCost'] + df['TotalStorageCost']
df['SGA'] = np.random.uniform(0.02, 0.05) * df['TotalSalesValue']
df['GM'] = df['TotalSalesValue'] - df['TotalCOGS']
df['CBM'] = df['GM'] - df['TotalS&D']
df['CBMAI'] = df['CBM'] - df['IOC']
df['EBITAI'] = df['CBMAI'] - df['SGA']

In [16]:
df['FY'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.strftime('%m-%Y')
df.columns

Index(['OrderID', 'FacilityID', 'ShipToID', 'Date', 'SalesVolume', 'ProductID',
       'ShipToCountry_x', 'ActualPrice', 'ProductName', 'Category',
       'KGPerPallet', 'SalesUnit', 'KGPerSalesUnit', 'Price', 'COGS',
       'ShelfLife', 'ShipToParty', 'Account', 'Channel', 'Address_ShipTo',
       'City_ShipTo', 'ShipToCountry_y', 'Continent_ShipTo', 'Latitude_ShipTo',
       'Longitude_ShipTo', 'FacilityName', 'FacilityType', 'City_Facility',
       'Country_Facility', 'Latitude_Facility', 'Longitude_Facility',
       'CapacityInPallet', 'AnnualStorageCost', 'HandlingCostPerPallet',
       'TotalOverheadCost', 'OriginID', 'OriginCity', 'OriginCountry',
       'DestinationID', 'DestinationCity', 'DestinationCountry', 'DistanceKM',
       'CostPerPallet', 'RouteType', 'SalesVolumeKG', 'QuantityInMT',
       'RawPallets', 'PalletsShipped', 'OutboundTransportCost',
       'TotalHandlingCost', 'StorageCost_FacProd', 'OverheadCost_FacProd',
       'IOC_FacProd', 'TotalTransferCost_FacProd'

In [17]:
final_cols = ['OrderID', 'Date', 'FY', 'Month','FacilityID', 'FacilityName','FacilityType', 'OriginCity', 'Country_Facility', 'ShipToID', 'Channel', 'ShipToParty','Account', 'DestinationCity', 'ShipToCountry_x', 
              'ProductID', 'ProductName','Category', 'QuantityInMT', 'PalletsShipped', 'TotalSalesValue', 'TotalCOGS', 
              'TotalStorageCost', 'TotalHandlingCost', 'TotalTransportCost', 'IOC', 'TotalS&D', 'GM', 'CBM', 'CBMAI', 'SGA', 'EBITAI']

final_df = df[final_cols]

In [18]:
final_df = final_df.rename(columns={
    'ShipToCountry_x': 'ShipToCountry'
})

In [19]:
# 1. Define the grouping keys
group_cols = ['FacilityID', 'ShipToID', 'ProductID', 'Month']

# 2. Define the columns to drop (we won't include them in the aggregation)
drop_cols = ['OrderID', 'Date']

# 3. Explicitly list the numerical columns that should be SUMMED
sum_cols = [
    'SalesVolume', 'SalesVolumeKG', 'QuantityInMT',
    'TotalTransportCost', 'TotalHandlingCost', 'StorageCostAllocated', 
    'OverheadCostAllocated', 'TotalStorageCost', 'IOC', 'TotalSalesValue', 
    'TotalCOGS', 'TotalS&D', 'SGA', 'GM', 'CBM', 'CBMAI', 'EBITAI'
]

# 4. Dynamically build the aggregation dictionary
agg_dict = {}
for col in final_df.columns:
    if col in group_cols or col in drop_cols:
        continue # Skip grouping keys and dropped columns
    elif col in sum_cols:
        agg_dict[col] = 'sum'
    else:
        # All other dimensional columns (Names, Categories, Prices, Lat/Long, etc.)
        # will just take the 'first' occurrence.
        agg_dict[col] = 'first'

# 5. Perform the GroupBy
df_monthly = final_df.groupby(group_cols).agg(agg_dict).reset_index()

In [20]:
df_monthly.head()

,FacilityID,ShipToID,ProductID,Month,FY,FacilityName,FacilityType,OriginCity,Country_Facility,Channel,...,TotalStorageCost,TotalHandlingCost,TotalTransportCost,IOC,TotalS&D,GM,CBM,CBMAI,SGA,EBITAI
0,F01,ST0003,P007,01-2024,2024,Bavaria Dairy Plant,Plant,Wasserburg,Germany,Key Account,...,90.399999,32.50,376.450,420.444172,499.349999,4208.75,3709.400001,3288.955829,1022.734301,2266.221528
1,F01,ST0003,P007,01-2025,2025,Bavaria Dairy Plant,Plant,Wasserburg,Germany,Key Account,...,76.680249,29.25,338.805,356.634557,444.735249,3570.00,3125.264751,2768.630194,867.516829,1901.113365
2,F01,ST0003,P007,02-2024,2024,Bavaria Dairy Plant,Plant,Wasserburg,Germany,Key Account,...,38.903950,19.50,225.870,180.939592,284.273950,1811.25,1526.976050,1346.036459,440.137215,905.899244
3,F01,ST0003,P007,02-2025,2025,Bavaria Dairy Plant,Plant,Wasserburg,Germany,Key Account,...,228.913096,81.25,941.125,1064.659046,1251.288096,10657.50,9406.211904,8341.552858,2589.792887,5751.759971
4,F01,ST0003,P007,03-2024,2024,Bavaria Dairy Plant,Plant,Wasserburg,Germany,Key Account,...,108.818294,39.00,451.740,506.106394,599.558294,5066.25,4466.691706,3960.585312,1231.108441,2729.476870


In [21]:
df_monthly.to_csv('final_profit_and_loss_transactions.csv', index=False)